<h1>ASSIGNMENT 3 INSTRUCTIONS</h1>



**This assignment accomplishes the following objectives:**
1. In assignment 2, we learned two key strategies to improve the quality and success rate of agenticAI workflows: tool use and reflection. In this assignment, we will learn a third key strategy: planning.
2. We will also learn how to mix and match LLMs to optimize inference costs.
3. We will then put these ideas togetheer to create an agenticAI workflow for blogging: Planner(Groq) -> Researcher(OpenAI + web_search tool) -> Writer(OpenAI) -> Editor(Groq) -> Reviser(OpenAI)

**In order to complete the assignmet:**
1. Section 1: Assignment setup. Just read through to understand what's already available.
2. Section 2: Create the agenticAI workflow. Complete coding steps 2.1 through 2.10.

**To submit:**
1. Rename the assignment as Assignment3-\<asurite\>
2. Runtime -> Restart session and run all
3. Ctrl-s
4. File -> Download .ipynb
5. Submit on Canvas

# SECTION 1: ASSIGNMENT SETUP (JUST READ THROUGH)

### We'll first import the required libraries.

In [1]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# install the orchestration layer (aisuite) plus the provider SDKs (OpenAI, Groq)
# in Colab, this is typically needed once per new runtime
!pip install aisuite openai groq

### We'll now setup the API KEYS and Clients.

In [2]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

from google.colab import userdata
import os
import aisuite
from openai import OpenAI
from groq import Groq

# read the API_KEYs from Colab Secrets and expose it as an env variable
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# create the clients
openai_client = OpenAI()
groq_client = Groq()
aisuite_client = aisuite.Client()

# select the models you want to use
OPENAI_MODEL= "gpt-4.1-mini"
AISUITE_OPENAI_MODEL = "openai:"+OPENAI_MODEL
GROQ_MODEL = "llama-3.3-70b-versatile"
AISUITE_GROQ_MODEL = "groq:"+GROQ_MODEL


### The reponses we get back from LLMs aren't always well-formatted. We'll use this utility function to print LLM responses.

In [3]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# a nifty utility function to cleanly diplay your chat outputs
from IPython.display import display, HTML, Markdown

def show(title, text):
    display(HTML(f"""
     <h4>{title}</h4>
     <div style="white-space: pre-wrap; padding-left: 24px;">{text}</div>"""))

### Recall from Assignment 1 that an LLM only has direct access to the information it was trained on and is unaware of events that occurred after its knowledge cutoff date. As in Assignment 2, we can address this limitation by providing the OpenAI LLM access to OpenAI's built-in web_search tool.

In [4]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

import json

# use OpenAI's built-in web_search tool to perform web_search
def openai_web_search(query: str):
  """
    Perform a web search using OpenAI's built-in web_search tool
    and return raw results.
  """
  resp = openai_client.responses.create(
      model=OPENAI_MODEL,
      input=query,
      tools=[{"type": "web_search"}],
  )
  return json.loads(resp.model_dump_json())

### The run_openai_query is identical to what we used in Assignment 2. It accepts the system prompt, the user prompt, and a list of tools that it can invoke as needed. And it returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [5]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API, invoke tools as needed;
# return response and query details (usage stats and tools invoked)
def run_openai_query(system_prompt, user_prompt, tools=[]):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Supports tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to OpenAI-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_OPENAI_MODEL, # OpenAI-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      tools=tools,
      max_turns=3,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details


### The run_groq_query is similar to the run_openai_query, with one important difference: it does not accept/invoke any tools. It returns both the model's response and relevant query details (including usage statistics and any tools that were called).

In [6]:
##############################
##### DO NOT EDIT THIS CELL!!!
##############################

# used for wall-clock timing for request
import time

# run the system/user prompt using OpenAI API; DOES NOT support tools;
# return response and query details (usage stats and tools invoked)
def run_groq_query(system_prompt, user_prompt):
  """
    Execute the query, print the prompt, the response, and detailed usage stats.
    Does not support tools.
  """

  # start the time before the API call
  start_time = time.time()

  # make chat-completions request thro aisuite, routed to Groq-backed model
  completion = aisuite_client.chat.completions.create(
      model=AISUITE_GROQ_MODEL, # Groq-backed model
      messages=[
          {"role": "system", "content": system_prompt}, # system sets global behavior/instructions
           {"role": "user", "content": user_prompt} # user contains the actual task/prompt
      ],
      max_turns=1,
  )

  # compute end-to-end wall-clock time for API request
  elapsed_time = time.time() - start_time

  # response
  response = completion.choices[0].message.content.strip()

  # usage stats
  query_details = {}
  query_details["prompt_tokens"]=completion.usage.prompt_tokens
  query_details["completion_tokens"]=completion.usage.completion_tokens
  query_details["total_tokens"]=completion.usage.total_tokens
  if hasattr(completion.usage, 'total_time'):
    query_details["total_time"]=f"{completion.usage.total_time:.2f}s"
  else:
    query_details["total_time"]=f"{elapsed_time:.2f}s"

  # tools invoked
  tools_invoked=[]
  messages = completion.choices[0].intermediate_messages
  for m in messages or []:
    if hasattr(m, "tool_calls") and m.tool_calls:
      for t in m.tool_calls:
        tools_invoked.append(t.function.name)
  query_details["tools_invoked"]=tools_invoked

  return response, query_details

# SECTION 2: LEARN HOW PLANNING WORKS (COMPLETE STEPS 2.1 THROUGH 2.10)

### Agentic AI Workflow with Planning

In this section, we implement an Agentic AI workflow with planning.

In the **Planning step**, we use an LLM to decide on the sequence of steps required to accomplish a complex task.

Specifically, we will implement the following workflow to write a blog on any given topic:

- topic → **PLANNER (Groq no tools)** → plan  
- plan → **RESEARCHER (OpenAI + web_search)** → research  
- plan, research → **WRITER (OpenAI no tools)** → draft  
- plan, research, draft → **EDITOR (Groq no tools)** → feedback  
- draft, feedback → **REVISER (OpenAI no tools)** → blog  

By decomposing the task into well-defined roles, we can use different LLMs for different steps - since not every part of the workflow requires deep reasoning or tool use.

Ultimately, these agenticAI workflow design choices modularize agenticAI workflows, enable more efficient use of models, and help optimize the unit economics of AI-powered products.

> **NOTE:** You may need to iterate on steps multiple times before settling on the final version for submission.

### We'll start by specifying the topic on which we want the agentic AI workflow to generate a blog post. You are encouraged to test the workflow with different topics to ensure that it generalizes well. However, your **final submission must use the original topic** stated below.

In [7]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# This is the topic we are attempting to write a blog on.
# You MUST NOT change this topic for your final submission.
# However, feel free to test the planner->researcher->writer->editor->reviser agenticAI workflow
# with different topics to ensure your code in Steps 2.1 through 2.10 generalizes well.
topic = "Artificial Intelligence"

### [Complete Coding STEP 2.1 and STEP 2.2]:

You'll now define the **planner** agent: topic -> Planner (Groq no tools) -> plan

You'll first update the system_prompt (STEP 2.1) and user_prompt (STEP 2.2) as you see fit.

You'll then execute the cell and examine the plan.

In [8]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.1: You have been provided with a system_prompt for the planner.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the planner.
planner_system_prompt = (
  f"""
  You are an expert content strategist and planning agent.
  Your sole responsibility is to architect a comprehensive blueprint for a blog post.
  Focus on logical flow, audience alignment, and identifying necessary factual evidence.
  Do not write the blog itself.
  """
)

# STEP-2.2: You have been provided with a starter user_prompt for the planner.
#           It must take the blog topic specified by the user.
#           It must then generate a plan that includes at a minimum, the
#           blog title, target audience, goal, outline, and researh checklist.
#           Update the user_prompt with clear instructions on plan requirements.
planner_user_prompt = (
  f"""
  Blog topic: {topic}

  Please create a detailed plan for this blog post including:
    1. A compelling and SEO-friendly Blog Title.
    2. A specific Target Audience description (e.g., industry professionals, students, or hobbyists).
    3. The primary Goal or "key takeaway" of the post.
    4. A detailed Outline with an introduction, at least three main body sections, and a conclusion.
    5. A Research Checklist containing at least 5 specific data points, recent trends, or technical concepts that the Researcher should investigate.

  Be structured and concise. Do not write any blog content or prose.
  """
)

# Have Groq (no tools) create a plan.
plan, query_details = run_groq_query(planner_system_prompt, planner_user_prompt)

show("----- PLANNER (GROQ WITHOUT TOOLS) -----", "")
show("Topic", topic)
show("Plan", plan)

### [Complete Coding STEP 2.3 and STEP 2.4]:

Next, you'll define the **researcher** agent: plan -> Researcher (OpenAI + web_search) -> research

You'll first update the system_prompt (STEP 2.3) and user_prompt (STEP 2.4) as you see fit.

You'll then execute the cell and examine the research.

In [9]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.3: You have been provided with a system_prompt for the researcher.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the researcher.
researcher_system_prompt = (
  f"""
  You are a meticulous research assistant specializing in AI and business technology.
  Your goal is to use the web_search tool to find high-quality, up-to-date data.
  You must provide factual evidence, technical definitions, and current statistics.
  Do not write blog prose or introductions; provide only organized research notes.
  """
)

# STEP-2.4: You have been provided with a starter user_prompt for the researcher.
#           It must take the outline and research checklist specified in the plan.
#           It must provide research notes that include at a minimum,
#           facts, definitions/stats where relevant, and sources.
#           Update the user_prompt with clear instructions on research requirements.
researcher_user_prompt = (
  f"""
  Using the Blog Plan provided below, conduct a thorough web search to gather supporting evidence.

  Blog plan (outline + checklist): {plan}

  For each outline section and research checklist item, provide:
    1. Key facts and technical explanations.
    2. At least two relevant, recent statistics or industry trends from 2025-2026.
    3. Proper citations or URLs for every piece of information found.
    4. Clarification of any complex terminology mentioned in the outline.
    5. A brief summary of expert opinions or case studies related to the topic.

  Return concise, bulleted research notes only. Do not attempt to write the blog draft.
  """
)

# Have OpenAI (with web_search) do the research.
research, query_details = run_openai_query(researcher_system_prompt, researcher_user_prompt, tools=[openai_web_search])

show("----- RESEARCHER (OPENAI WITH WEB_SEARCH) -----", "")
show("Research", research)

### [Complete Coding STEP 2.5 and STEP 2.6]:

Next, you'll define the **writer** agent: plan, research -> Writer (OpenAI no tools) -> draft

You'll first update the system_prompt (STEP 2.5) and user_prompt (STEP 2.6) as you see fit.

You'll then execute the cell and examine the draft.

In [10]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.5: You have been provided with a system_prompt for the writer.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the writer.
writer_system_prompt = (
  f"""
  You are a professional blog writer and subject matter expert.
  Your goal is to transform structured plans and raw research into a polished, engaging, and informative blog post.
  Maintain a tone that is appropriate for the target audience and ensure the narrative flow is logical and persuasive.
  Focus entirely on writing the content; do not provide meta-commentary or critiques of your work.
  """
)

# STEP-2.6: You have been provided with a starter user_prompt for the writer.
#           It must take the audience, goal, and outline specified in the plan,
#           and the research and sources specified in the research.
#           It must write a one-page blog draft.
#           Update the user_prompt with clear instructions on how to use plan and research to write.
writer_user_prompt = (
  f"""
  Using the structural plan and the detailed research notes provided below, write a complete first draft of the blog post.

  Audience, goal, and outline from planner: {plan}
  Research and sources from researcher: {research}

  Requirements for the blog:
    1. Follow the outline provided in the plan strictly, ensuring all main sections are covered.
    2. Incorporate the specific facts, statistics, and citations found by the researcher to ground the post in evidence.
    3. Adapt the voice and vocabulary to speak directly to the specified target audience.
    4. Ensure smooth transitions between paragraphs and a clear, compelling introduction and conclusion.
    5. The draft should be approximately one page (500-700 words) in length.

  Write only the blog content.
  """
)

# Have OpenAI (no tools) write a draft.
draft, query_details = run_openai_query(writer_system_prompt, writer_user_prompt, tools=[])

show("----- WRITER (OPENAI WITHOUT TOOLS) -----", "")
show("Draft", draft)

### [Complete Coding STEP 2.7 and STEP 2.8]:

Next, you'll define the **editor** agent: plan, research, draft -> Editor (Groq no tools) -> feedback

You'll first update the system_prompt (STEP 2.7) and user_prompt (STEP 2.8) as you see fit.

You'll then execute the cell and examine the feedback.

In [11]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.7: You have been provided with a system_prompt for the editor.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the editor.
editor_system_prompt = (
  f"""
  You are a senior executive editor with an eye for technical accuracy and narrative impact.
  Your role is to critically evaluate blog drafts against their original strategic plans and research notes.
  You provide rigorous, constructive feedback and objective scoring.
  Do not rewrite the text yourself; your goal is to guide the reviser with high-quality feedback.
  """
)

# STEP-2.8: You have been provided with a starter user_prompt for the editor.
#           It must take the plan, research, and draft.
#           It must evaluate based on a list of critera specified by you, and
#           provide a score for each criteria and actionable improvement suggestions.
#           Update the user_prompt with a list of criteria and instructions for scoring and improvements.
editor_user_prompt = (
  f"""
  Please perform a comprehensive editorial review of the following draft.

  Original audience, goal, and outline from planner: {plan}
  Research and sources from researcher: {research}
  Draft from writer: {draft}

  Evaluate the draft based on these 5 criteria:
    1. Plan Alignment: Does the draft follow the specified outline and meet the stated goal?
    2. Fact Integration: Are the statistics and research notes from the researcher used accurately?
    3. Audience Resonance: Is the tone and complexity level appropriate for the target audience?
    4. Structural Flow: Are the transitions logical and is the narrative cohesive?
    5. Clarity & Impact: Is the writing concise and the call-to-action (if any) clear?

  Provide your feedback in the following format:
    1. A score from 1-10 for each of the 5 criteria above.
    2. A brief justification for each score.
    3. A list of at least 3 actionable improvement suggestions for the next revision.
    4. Identification of any factual inconsistencies or missing research points.
    5. Final "Pass/Fail" recommendation for publication.

  Do not rewrite the blog content. Provide only the evaluation and feedback.
  """
)

# Have Groq (no tools) provide feedback on the draft.
feedback, query_details = run_groq_query(editor_system_prompt, editor_user_prompt)

show("----- EDITOR (GROQ WITHOUT TOOLS) -----", "")
show("Feedback", feedback)

### [Complete Coding STEP 2.9 and STEP 2.10]:

Next, you'll define the **reviser** agent: draft, feedback -> Reviser (OpenAI no tools) -> blog

You'll first update the system_prompt (STEP 2.9) and user_prompt (STEP 2.10) as you see fit.

You'll then execute the cell and examine the blog.

In [12]:
#########################################################################
##### ONLY CODE AS INDICATED - DO NOT ALTER THE CELL IN  ANY OTHER WAY!!!
#########################################################################

# STEP-2.9: You have been provided with a system_prompt for the reviser.
#           Feel free to update the system_prompt if required.
#           But you MUST NOT change the role of the reviser.
reviser_system_prompt = (
  f"""
  You are an expert senior editor and blog reviser.
  Your goal is to take a first draft and transform it into a publication-ready masterpiece by strictly following editorial feedback.
  You excel at smoothing out transitions, correcting factual inaccuracies, and sharpening the prose to perfectly match the target audience's expectations.
  """
)

# STEP-2.10: You have been provided with a starter user_prompt for the reviser.
#           It must take the draft and feedback.
#           It must return a revised one-page blog.
#           Update the user_prompt with clear instructions on how to use draft and feedback to write.
reviser_user_prompt = (
  f"""
  Please produce the final version of the blog post based on the draft and feedback provided below.

  Original blog draft: {draft}
  Editor feedback: {feedback}

  Instructions for the revision:
    1. Address every specific suggestion and critique provided in the editor's feedback.
    2. Improve the flow and narrative connection between sections to ensure a seamless reading experience.
    3. Verify that all technical facts and statistics are presented clearly and accurately.
    4. Ensure the tone is perfectly calibrated for the target audience identified in the initial plan.
    5. Deliver a polished, professional, one-page blog post that represents the highest quality of AI-assisted writing.

  Return only the final blog post content.
  """
)

# Have OpenAI (no tools) creat the blog.
blog, query_details = run_openai_query(reviser_system_prompt, reviser_user_prompt, tools=[])

show("----- REVISER (OPENAI WITHOUT TOOLS) -----", "")
show("Blog", blog)